# Separate import accounts

This companion runs on the packaged SUT and IOT fixtures. The guide explains the import equations and configuration choices.

In [ ]:
import mario
import numpy as np
import pandas as pd
from pathlib import Path
from tempfile import TemporaryDirectory

In [ ]:
source = mario.load_test("SUT")
regional = source.to_single_region(
    source.regions[0], imports_by="commodity", trade_resolution="by_region",
    import_representation="separate", inplace=False,
)
regional.configuration.to_dict()

In [ ]:
blocks = regional.query(["Va", "Vc", "IMP_a", "IMP_c", "IMP_Y"])
assert regional.IMP_a.index.equals(regional.IMP_Y.index)
assert "imports" not in regional.factors
np.testing.assert_allclose(
    regional.imp_a.to_numpy() * regional.Xa.to_numpy().ravel(), regional.IMP_a,
)

In [ ]:
combined = regional.to_import_representation("factor-accounts", inplace=False)
pd.testing.assert_frame_equal(regional.p, combined.p)
restored = combined.to_import_representation("separate", inplace=False)
pd.testing.assert_frame_equal(restored.IMP_a, regional.IMP_a)

In [ ]:
with TemporaryDirectory() as folder:
    folder = Path(folder)
    regional.export_sets(path=folder / "sets.xlsx")
    regional.to_parquet(path=folder, flows=True, coefficients=False)
    reloaded = mario.parse_from_parquet(str(folder), table="SUT", mode="flows")
    assert reloaded.configuration == regional.configuration
    actual = reloaded.IMP_a.reindex(index=regional.IMP_a.index, columns=regional.IMP_a.columns)
    np.testing.assert_allclose(actual, regional.IMP_a)

In [ ]:
iot_source = mario.load_test("IOT")
iot = iot_source.to_single_region(
    iot_source.regions[0], imports_by="sector",
    import_representation="separate", inplace=False,
)
import_coefficients = iot.imp
import_requirements = iot.im
assert import_coefficients.index.equals(iot.IMP.index)